# SupplyMind AI — Focused EDA

This notebook explores only model-relevant questions after the target,
timestamp, leakage columns, and eligible features have been confirmed.

In [ ]:
# -------------------
# Imports
# -------------------

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from supplymind.features.predictions.application.dataset import (
    load_tabular_dataset,
    normalize_column_names,
)
from supplymind.features.predictions.ml.cleaning import clean_shipment_data

In [ ]:
# -------------------
# Configuration
# -------------------

DATASET_PATH = Path("../data/raw/syndelay/YOUR_DATASET_FILE.csv")
TARGET_COLUMN = "REPLACE_WITH_CONFIRMED_TARGET"
TIMESTAMP_COLUMN = "REPLACE_WITH_CONFIRMED_TIMESTAMP"

In [ ]:
# -------------------
# Load and clean
# -------------------

df = load_tabular_dataset(DATASET_PATH)
df = normalize_column_names(df)
df = clean_shipment_data(df)

df.head()

In [ ]:
# -------------------
# Target distribution
# -------------------

target_counts = df[TARGET_COLUMN].value_counts(dropna=False).sort_index()
target_rates = df[TARGET_COLUMN].value_counts(
    normalize=True,
    dropna=False,
).sort_index()

display(pd.DataFrame({
    "count": target_counts,
    "rate": target_rates,
}))

target_counts.plot(kind="bar", title="Target distribution")
plt.xlabel(TARGET_COLUMN)
plt.ylabel("Rows")
plt.show()

In [ ]:
# -------------------
# Missing-value analysis
# -------------------

missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)

missing.head(30)

In [ ]:
# -------------------
# Numerical distributions
# -------------------

numerical_columns = df.select_dtypes(include="number").columns.tolist()
numerical_columns = [
    column for column in numerical_columns
    if column != TARGET_COLUMN
]

df[numerical_columns].describe().T

In [ ]:
# -------------------
# Categorical cardinality
# -------------------

categorical_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_profile = pd.DataFrame({
    "column": categorical_columns,
    "unique_count": [
        df[column].nunique(dropna=True)
        for column in categorical_columns
    ],
    "missing_rate": [
        df[column].isna().mean()
        for column in categorical_columns
    ],
}).sort_values("unique_count", ascending=False)

categorical_profile

In [ ]:
# -------------------
# Temporal target trend
# -------------------

temporal_df = df[[TIMESTAMP_COLUMN, TARGET_COLUMN]].copy()
temporal_df[TIMESTAMP_COLUMN] = pd.to_datetime(
    temporal_df[TIMESTAMP_COLUMN],
    errors="coerce",
)
temporal_df = temporal_df.dropna(subset=[TIMESTAMP_COLUMN])
temporal_df["period"] = temporal_df[TIMESTAMP_COLUMN].dt.to_period("M")

monthly_delay_rate = (
    temporal_df.groupby("period", observed=True)[TARGET_COLUMN]
    .mean()
)

monthly_delay_rate.plot(
    title="Monthly delayed-shipment rate",
    figsize=(12, 4),
)
plt.ylabel("Delay rate")
plt.show()

## EDA conclusions

Record only decisions that affect the production pipeline:

- confirmed class imbalance
- missing-value strategy
- high-cardinality columns
- suspicious target proxies
- temporal drift
- impossible or invalid values
- candidate features to keep
- columns to exclude and why